# IMU 数据预处理流水线

**功能总览**

| Cell | 功能 | 说明 |
|------|------|------|
| Cell 0 | 全局参数配置 | 集中定义所有路径与参数，修改配置只需编辑本 cell |
| Cell 1 | 视频配准 | 加载 IMU 原始数据（变量 `raw`），启动交互式 GUI 与视频对齐，获得时间偏移 `t_diff` |
| Cell 2 | 数据处理与导出 | **复用 Cell 1 的 `raw` 与 `t_diff`**（或用 `T_DIFF_MANUAL` 手动指定），执行完整预处理并保存 CSV |

**依赖文件**（与本 notebook 同目录）：
- `imu_processor.py` — 数据导入与预处理核心库（**无上采样，输出 ~99Hz、8 列**）
- `aligner.py` — IMU-视频时间配准 GUI（支持传入预加载的 `raw`）
- `imu_utils.py` — `save_processed_csv` 等辅助函数

**输出格式（8 列）**：`time, ax, ay, az, wx, wy, wz, interp_mask`
- 采样率 ≈ `_align_imu` 估计的片上采样率（~99Hz，非整数秒时间戳）
- 信号已做 20Hz 低通滤波，**不再做 EKF 重力分离**（与早期粗分类版本不同）
- 后续切分（滑动窗口等）请基于**采样点数**而非秒；切分时同步切分 `time` 列

**典型流程**：先运行 Cell 0 配置参数 → 运行 Cell 1 完成配准（GUI 关闭后自动记录 `t_diff`）→ 运行 Cell 2 导出数据。
若已有已知的 `t_diff`，可跳过 Cell 1，在 Cell 0 中设置 `T_DIFF_MANUAL` 后直接运行 Cell 2。

**一般来说，一段视频对应2段IMU，可以选择两段IMU中同步信号更明显的来配准，配准一次，t_diff可以复用**

## Cell 0 — 全局参数配置 (Global Paths & Parameters)

In [2]:
# ============================================================
# Cell 0: 全局参数配置 (Global Paths & Parameters)
# 功能: 集中定义所有路径与参数, 供 Cell 1 / Cell 2 共用
# ============================================================

# --------------------- 路径配置 ---------------------
VIDEO_PATH  = r"\\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test3_vs_Tool1\data2.mp4"     # 视频文件路径 (Cell 1 配准用)
IMU_DIR     = r"\\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test3_vs_Tool1\ab"            # IMU 原始 CSV 目录 (只加载一次)
OUTPUT_PATH = r"\\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test3_tool1_ab.csv" # 输出 CSV 路径 (Cell 2 保存用)

# --------------------- Cell 1 (配准) 专用参数 ---------------------
IMU_DURATION = 280.0  # GUI 中仅显示 IMU 前 N 秒 (加速显示)
VIEW_WINDOW  = 30.0   # IMU 曲线初始视窗宽度 (s)

# --------------------- Cell 2 (处理导出) 专用参数 ---------------------
# 输出采样率 ≈ 片上估计 ~99Hz (由 _align_imu 给出, 不做上采样)
T_DIFF_MANUAL = None  # 手动指定 t_diff (秒); None = 使用 Cell 1 的配准结果
APPLY_LPF     = True  # 是否低通滤波
LPF_FC        = 20.0  # 低通滤波截止频率 (Hz)
# ----------------------------------------------------

print("配置完成:")
print(f"  VIDEO_PATH  = {VIDEO_PATH}")
print(f"  IMU_DIR     = {IMU_DIR}")
print(f"  OUTPUT_PATH = {OUTPUT_PATH}")
print(f"  T_DIFF_MANUAL = {T_DIFF_MANUAL}")

配置完成:
  VIDEO_PATH  = \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test3_vs_Tool1\data2.mp4
  IMU_DIR     = \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test3_vs_Tool1\ab
  OUTPUT_PATH = \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\IMU_Data\processed\test3_tool1_ab.csv
  T_DIFF_MANUAL = None


## Cell 1 — 视频配准 (Video-IMU Time Alignment)

In [ ]:
# ============================================================
# Cell 1: 视频配准 (Video-IMU Time Alignment)
# 功能: 加载 IMU 原始数据 -> 启动交互式 GUI 与视频对齐 -> 获得 t_diff
# 输入: Cell 0 的 VIDEO_PATH / IMU_DIR / LPF_FC / IMU_DURATION / VIEW_WINDOW
# 输出: raw (原始数据) / t_diff (时间偏移) — Cell 2 复用
# ============================================================

%matplotlib tk
from imu_processor import data_import
from aligner import IMUVideoAligner

# raw 只在此处加载一次, Cell 2 直接复用同一变量
raw = data_import(IMU_DIR, verbose=True)

aligner = IMUVideoAligner(
    VIDEO_PATH,
    raw_data=raw,               # 传入预加载数据, 不重复读取磁盘
    lpf_fc=LPF_FC,
    imu_duration=IMU_DURATION,
    view_window=VIEW_WINDOW,
)
aligner.show()   # 弹出 GUI 窗口; 关闭窗口后代码继续往下执行

In [ ]:
# 记录配准结果 (GUI 中 "Save t_diff" 同时会写入视频旁的 t_diff.txt)
t_diff = aligner.offset
print(f"\n配准结果: t_diff = {t_diff:.6f} s  (Cell 2 将默认使用该值)")

## Cell 2 — 数据处理与导出 (Data Processing & Export)

In [ ]:
# ============================================================
# Cell 2: 数据处理与导出 (Data Processing & Export)
# 功能: 复用 Cell 1 的 raw 与 t_diff -> 完整预处理 -> 保存 CSV
# 输入: Cell 0 的全部参数; Cell 1 的 raw / t_diff
# 输出: ~99Hz, 8 列 CSV (time, ax, ay, az, wx, wy, wz, interp_mask)
# ============================================================

from imu_processor import data_import, data_preprocess
from imu_utils import save_processed_csv

# t_diff 来源: Cell 0 的 T_DIFF_MANUAL 优先, 否则使用 Cell 1 配准结果
if T_DIFF_MANUAL is not None:
    t_diff = T_DIFF_MANUAL
    print(f"使用手动 t_diff = {t_diff:.6f} s (Cell 0 配置)")
elif "t_diff" not in globals():
    raise RuntimeError("未检测到 t_diff: 请先运行 Cell 1 完成配准, "
                       "或在 Cell 0 中设置 T_DIFF_MANUAL")
else:
    print(f"使用 Cell 1 配准 t_diff = {t_diff:.6f} s")

# 若跳过了 Cell 1 (raw 未定义), 在此处加载; 否则复用 Cell 1 的 raw
if "raw" not in globals():
    print("[Cell 2] 未检测到 Cell 1 的 raw, 从磁盘加载...")
    raw = data_import(IMU_DIR, verbose=True)

# 完整预处理 (复用 raw, 不重复读取)
# 输出 ~99Hz, 8 列: time, ax, ay, az, wx, wy, wz, interp_mask
proc, info = data_preprocess(
    raw,
    t_diff=t_diff,
    apply_lpf=APPLY_LPF,
    lpf_fc=LPF_FC,
    return_drift=True,
)

print(f"proc shape: {proc.shape}")
print(f"fs = {info['fs']:.2f} Hz, drift = {info['drift']:.3f}%, "
      f"dropped = {info['dropped_samples']}, interpolated = {info['interpolated_samples']}")

# 保存 CSV (8 通道: time, ax, ay, az, wx, wy, wz, interp_mask)
save_processed_csv(proc, OUTPUT_PATH)